# Synapse: AI-Native Engineering Mindset Challenge

Welcome to your first real-world AI-native engineering challenge! 

In this lab, we are going to bridge the gap between **traditional software engineering** and **AI engineering**. 

### 🎯 The Goal
You aren't just "chatting with an AI." You are building an **Automated Support Ticket Router**. This system must read messy customer support emails, summarize them, and extract structured metadata (department and priority) so the ticket can be routed to the right team.

### 🧠 The Mindset Shift
By the end of this notebook, you will have proved three things to yourself:
1.  **The Runtime Lens:** LLMs are slow and expensive compared to traditional code. You must measure latency and cost.
2.  **The Deterministic Boundary:** LLMs are chaotic components. You must build strict "guardrails" in your Python code to handle their failures.
3.  **The Eval Primer:** You don't test AI by looking at one answer; you test it against a set of scenarios.

## 🛠️ Step 0: Resilient Setup

To run this lab, you need an **OpenAI API Key**.

1.  **Get a Key:** Go to the [OpenAI API Dashboard](https://platform.openai.com/).
2.  **Add Credits:** AI engineering requires **API Credits**. Note that "ChatGPT Plus" is a consumer subscription and does *not* give you API access. You must add at least $5 to your API balance.
3.  **Keep it Secret:** Never hardcode your key in a file you might commit to Git.

In [1]:

import getpass
import os
import time
import json
try:
    from dotenv import load_dotenv
    load_dotenv()  # This loads the OPENAI_API_KEY from your .env file
    from openai import OpenAI, AuthenticationError, RateLimitError
except ImportError:
    !pip install -q openai python-dotenv
    from dotenv import load_dotenv
    load_dotenv()
    from openai import OpenAI, AuthenticationError, RateLimitError

# Securely input API key
if not os.getenv("OPENAI_API_KEY"):
    
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

try:
    client = OpenAI()
    # Test the client with a tiny call to ensure the key is valid and funded
    client.models.list()
    print("✅ API Client initialized successfully!")
except AuthenticationError:
    print("❌ ERROR: Invalid API Key. Please double-check your key.")
except RateLimitError:
    print("❌ ERROR: You have hit a rate limit or have 0 credits. Check your billing at platform.openai.com.")
except Exception as e:
    print(f"❌ ERROR: An unexpected error occurred: {e}")

✅ API Client initialized successfully!


## 🏁 Part 1: The Runtime Lens (Instrumentation)

Traditional functions take microseconds. LLM calls take **seconds**. 
Traditional functions are nearly free. LLM calls cost **money** per transaction.

In this part, you will summarize a support ticket and calculate the exact **latency** and **cost**.

In [2]:
MESSY_TICKET = """
Subject: HELP!!! I can't log in and I'm losing money!!!
Body: Hey, I've been trying to get into my billing dashboard for three hours. 
It just says 'Server Error'. This is unacceptable. I have a big client meeting 
in an hour and I need to download my invoice. My account email is boss@startup.io. 
Fix this now or I'm cancelling my subscription!!
"""

def instrumented_summary(text):
    start_time = time.time()
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Summarize support tickets in one sentence."},
            {"role": "user", "content": text}
        ]
    )
    
    end_time = time.time()
    
    # Metrics Calculation
    latency = end_time - start_time
    usage = response.usage
    
    # Prices for gpt-4o-mini: $0.15 / 1M input tokens, $0.60 / 1M output tokens
    cost = (usage.prompt_tokens * 0.00000015) + (usage.completion_tokens * 0.00000060)
    
    print(f"Summary: {response.choices[0].message.content}")
    print(f"--- Runtime Stats ---")
    print(f"Latency: {latency:.2f} seconds")
    print(f"Estimated Cost: ${cost:.6f}")
    
    return response

instrumented_summary(MESSY_TICKET)

Summary: The user is unable to log into their billing dashboard due to a server error, urgently needing access to download an invoice before an important client meeting, and is threatening to cancel their subscription if not resolved quickly.
--- Runtime Stats ---
Latency: 2.95 seconds
Estimated Cost: $0.000040


ChatCompletion(id='chatcmpl-DgBKTIguZP5DFzAkua41u5FWnLSx1', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The user is unable to log into their billing dashboard due to a server error, urgently needing access to download an invoice before an important client meeting, and is threatening to cancel their subscription if not resolved quickly.', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None, annotations=[]))], created=1778945697, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_fbf6c0be0a', usage=CompletionUsage(completion_tokens=41, prompt_tokens=100, total_tokens=141, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

## 🛡️ Part 2: The Deterministic Boundary

Now, we need to extract structured data (JSON) to route the ticket. 
But there's a problem: **Models are probabilistic.** They might return broken JSON, or they might invent a department that doesn't exist.

**Your Task:** Write a "Defensive Boundary" function that attempts to parse the LLM response but falls back to a safe, deterministic default if the AI fails.

In [3]:
VALID_DEPARTMENTS = ["billing", "technical", "sales"]
DEFAULT_ROUTE = {"department": "human_review", "priority": "high", "is_ai_failure": True}

def parse_and_route(llm_output):
    """
    IMPLEMENT THIS FUNCTION.
    Goal: Ensure probabilistic AI output is converted to deterministic software logic.
    """
    # --- YOUR CODE HERE ---
    
    return None

# --- DO NOT EDIT BELOW THIS LINE (Local Test) ---
test_response = '{"department": "billing", "priority": "high"}'
print(f"Test 1 (Valid JSON): {parse_and_route(test_response)}")

bad_response = "Sure! The department is: finance" 
print(f"Test 2 (AI Filler/Bad Dept): {parse_and_route(bad_response)}")

Valid Response Result: {'department': 'billing', 'priority': 'high'}
Bad Response Result: {'department': 'human_review', 'priority': 'high', 'is_ai_failure': True}


## 📊 Part 3: The "Eval" Primer

In AI engineering, you don't test code with a single unit test. You test it against a **Scorecard** of scenarios.

We will now run your `parse_and_route` function against three different real-world scenarios to see if your boundary holds up.

In [ ]:
SCENARIOS = [
    {
        "name": "Happy Path",
        "input": '{"department": "technical", "priority": "medium"}',
        "expected_dept": "technical"
    },
    {
        "name": "Hallucination (Invalid Dept)",
        "input": '{"department": "account_recovery", "priority": "high"}', # account_recovery is not in VALID_DEPARTMENTS
        "expected_dept": "human_review"
    },
    {
        "name": "Malformed JSON (AI Filler)",
        "input": "I have analyzed the ticket and it belongs to billing.", # Not JSON
        "expected_dept": "human_review"
    }
]

def run_scorecard():
    print("--- AI-Native Scorecard ---\n")
    passed = 0
    for s in SCENARIOS:
        result = parse_and_route(s["input"])
        actual_dept = result.get("department")
        
        status = "✅ PASS" if actual_dept == s["expected_dept"] else "❌ FAIL"
        if status == "✅ PASS": passed += 1
        
        print(f"{status} | Scenario: {s['name']}")
        print(f"      Input: {s['input'][:50]}...")
        print(f"      Resulting Dept: {actual_dept}\n")
    
    print(f"Total Score: {passed}/{len(SCENARIOS)}")
    if passed == len(SCENARIOS):
        print("\n🚀 Mindset Unlocked: You have successfully built your first deterministic AI boundary!")

run_scorecard()

## 📱 Return to Synapse

Congratulations on completing your first lab! 

Scan the QR code below with your phone to jump directly back to the Synapse app and start **Course 2: Conceptual LLM Mechanics**.

In [ ]:
import qrcode
from IPython.display import display

# The deep link to the next module in the Synapse app
NEXT_MODULE_URL = "https://ai-in-a-shell.com/learn/02-conceptual-llm-mechanics"

def generate_handoff_qr(url):
    qr = qrcode.QRCode(version=1, box_size=10, border=5)
    qr.add_data(url)
    qr.make(fit=True)
    img = qr.make_image(fill_color="black", back_color="white")
    display(img)

generate_handoff_qr(NEXT_MODULE_URL)